In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%cd /Workspace/Users/parkjea@pennmedicine.upenn.edu/
!pip install google-generativeai
!pip install decord
from json_utils import *

In [0]:
from dataset.build_charades import *
split_charadessta_train_val(
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/charades_sta_train.json",
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/charades_sta_train_split.json",
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/charades_sta_val_split.json",
    val_ratio=0.1,
    seed=123,
)

precompute_charades_text_emb(
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/charades_sta_train_split.json",
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/Extracted/clip-vit-base-patch32-json/train_split.npy",
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/Extracted/clip-vit-base-patch32-json/train_split_tokens.npy",
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/Extracted/clip-vit-base-patch32-json/train_split_mask.npy",
    model_name="openai/clip-vit-base-patch32",
    batch_size=256,
    device="cuda",
    fp16=True,)
precompute_charades_text_emb(
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/charades_sta_val_split.json",
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/Extracted/clip-vit-base-patch32-json/val_split.npy",
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/Extracted/clip-vit-base-patch32-json/val_split_tokens.npy",
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/Extracted/clip-vit-base-patch32-json/val_split_mask.npy",
    model_name="openai/clip-vit-base-patch32",
    batch_size=256,
    device="cuda",
    fp16=True,)
precompute_charades_text_emb(
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/charades_sta_test.json",
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/Extracted/clip-vit-base-patch32-json/test.npy",
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/Extracted/clip-vit-base-patch32-json/test_tokens.npy",
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/Extracted/clip-vit-base-patch32-json/test_mask.npy",
    model_name="openai/clip-vit-base-patch32",
    batch_size=256,
    device="cuda",
    fp16=True,)



In [0]:
tr_ds, val_ds, test_ds, tr_loader, val_loader, test_loader = load_dataset()

In [0]:
!ls /local_disk0/embed_cache/KZK6W 

# Sanity Check (Dataset Load and Process)

In [0]:
import torch
import numpy as np

batch = next(iter(val_loader))   # loader = DataLoader(..., collate_fn=...)

print("keys:", batch.keys())
print("video_emb:", batch["video_emb"].shape, batch["video_emb"].dtype)   # (B, Tmax, D)
print("video_mask:", batch["video_mask"].shape, batch["video_mask"].dtype)
print("text_emb:", batch["text_emb"].shape, batch["text_emb"].dtype)      # (B, D)
print("lengths:", batch["lengths"][:8])

# check first sample metadata
j = 0
print("\nSample 0:")
print("video_id:", batch["video_id"][j])
print("query:", batch["query"][j])
print("start/end:", batch["start_sec"][j], batch["end_sec"][j])
print("i0/i1:", batch["i0"][j], batch["i1"][j])
print("valid frames:", int(batch["video_mask"][j].sum().item()))

In [0]:
# Text Embedding Sanity Check
import torch
from transformers import CLIPModel, CLIPProcessor

@torch.no_grad()
def compare_text_emb(batch, clip_model, clip_processor, device="cuda"):
    """
    batch should contain:
      - batch["query"] : list[str]
      - batch["text_emb"] : (B, D) torch or np
    Returns cosine similarity per sample.
    """
    queries = batch["query"]
    pre = batch["text_emb"]

    # move precomputed to torch on device
    pre = torch.as_tensor(pre, dtype=torch.float32, device=device)
    pre = pre / pre.norm(dim=-1, keepdim=True).clamp(min=1e-6)

    # compute CLIP text embeddings now
    inputs = clip_processor(text=queries, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    cur = clip_model.get_text_features(**inputs)
    cur = cur / cur.norm(dim=-1, keepdim=True).clamp(min=1e-6)
    cur = cur.float()

    # cosine similarity (since normalized, dot product)
    cos = (cur * pre).sum(dim=-1)  # (B,)
    return cos, cur, pre

# ---- usage ----
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "openai/clip-vit-base-patch32"

clip_model = CLIPModel.from_pretrained(model_name).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(model_name)

cos, cur, pre = compare_text_emb(batch, clip_model, clip_processor, device=device)

print("cosine stats:",
      "min", cos.min().item(),
      "mean", cos.mean().item(),
      "max", cos.max().item())

# show a few examples
for i in range(min(5, len(cos))):
    print(f"\n[{i}] cos={cos[i].item():.6f}")
    print("query:", batch["query"][i])


In [0]:
import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from decord import VideoReader, cpu

@torch.no_grad()
def sanity_check_random_frames(
    batch,
    raw_video_root: str,
    embed_cache_root: str = "/local_disk0/embed_cache",
    video_ext: str = ".mp4",
    clip_model=None,          # transformers.CLIPModel
    clip_processor=None,      # transformers.CLIPProcessor
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
    k: int = 4,               # how many samples from batch to check
    seed: int = 0,
    show: bool = False,       # set True to display frames
):
    """
    Expects batch (from collate_charadessta_pad) to contain:
      batch["video_id"] : list[str]
      batch["i0"] : list[int]
      batch["i1"] : list[int]
      batch["video_emb"] : (B, Tmax, D) torch.float32
      batch["video_mask"] : (B, Tmax) bool
      batch["query"] : list[str]

    Requires per-video files in embed_cache_root/<video_id>/:
      - frame_indices.npy  (T,)  original frame indices
      - clip_embeddings_fp16.npy (T, D)  (optional if you want to load directly from disk)
      (but we can just use batch["video_emb"] for the segment embedding)
    """
    assert clip_model is not None and clip_processor is not None, "Pass CLIPModel and CLIPProcessor."
    clip_model = clip_model.to(device).eval()

    rng = random.Random(seed)

    B = len(batch["video_id"])
    idxs = list(range(B))
    rng.shuffle(idxs)
    idxs = idxs[: min(k, B)]

    raw_video_root = Path(raw_video_root)
    embed_cache_root = Path(embed_cache_root)

    video_emb = batch["video_emb"]  # torch (B, Tmax, D)
    video_mask = batch["video_mask"]  # torch (B, Tmax)

    results = []

    for b in idxs:
        vid = batch["video_id"][b]
        q = batch["query"][b]
        i0 = int(batch["i0"][b])
        i1 = int(batch["i1"][b])

        # valid length in padded tensor (segment length)
        L = int(video_mask[b].sum().item())
        if L <= 0:
            results.append((vid, None, None, "EMPTY_SEG"))
            continue

        # pick a random position within the segment portion [0, L)
        pos = rng.randrange(L)

        # global timestep in the per-video embedding sequence
        t_global = i0 + pos

        # load mapping from timestep -> original video frame index
        vdir = embed_cache_root / vid
        frame_idx_path = vdir / "frame_indices.npy"
        if not frame_idx_path.exists():
            raise FileNotFoundError(f"Missing {frame_idx_path}. Did you cache/copy embeddings locally?")

        frame_idx = np.load(str(frame_idx_path))  # (T,)
        if t_global < 0 or t_global >= len(frame_idx):
            results.append((vid, None, None, f"t_global out of range: {t_global}/{len(frame_idx)}"))
            continue

        src_frame = int(frame_idx[t_global])

        # decode the actual frame from video
        # Adjust naming if your videos are stored differently.
        video_path = raw_video_root / f"{vid}{video_ext}"
        if not video_path.exists():
            # common Databricks path form for volumes is /dbfs/Volumes/...
            # if you passed /Volumes/... and decord can't read, try raw_video_root="/dbfs/Volumes/..."
            raise FileNotFoundError(f"Video file not found: {video_path}")

        vr = VideoReader(str(video_path), ctx=cpu(0))
        src_frame = max(0, min(src_frame, len(vr) - 1))
        frame = vr[src_frame].asnumpy()  # (H,W,3) uint8 RGB
        img = Image.fromarray(frame)

        # compute CLIP image embedding from decoded frame
        inputs = clip_processor(images=[img], return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        cur = clip_model.get_image_features(**inputs)  # (1, D)
        cur = cur / cur.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        cur = cur[0].float()  # (D,)

        # get the saved embedding corresponding to this position from the batch tensor
        saved = video_emb[b, pos].to(device).float()
        saved = saved / saved.norm(dim=-1, keepdim=False).clamp(min=1e-6)

        cos = torch.dot(cur, saved).item()

        results.append((vid, src_frame, [i0, i1], cos, q))

        if show:
            try:
                import matplotlib.pyplot as plt
                plt.figure()
                plt.imshow(img)
                plt.axis("off")
                plt.title(f"{vid} frame={src_frame} cos={cos:.4f}\n{q}")
                plt.show()
            except Exception:
                pass

    # Print summary
    for vid, src_frame, time, cos, q in results:
        if cos is None:
            print(f"[{vid}] ERROR: {q}")
        else:
            print(f"[{vid}] src_frame={src_frame} cos={cos:.4f} | {q} | {time}")

    return results


In [0]:
from transformers import CLIPModel, CLIPProcessor
import torch

model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(model_name)
clip_processor = CLIPProcessor.from_pretrained(model_name)

#batch = next(iter(val_loader))

# raw videos directory example:
#   /Volumes/<catalog>/<schema>/<volume>/videos   OR   /dbfs/Volumes/<...>/videos
sanity_check_random_frames(
    batch,
    raw_video_root="/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/Charades_v1",
    embed_cache_root="/local_disk0/embed_cache",
    clip_model=clip_model,
    clip_processor=clip_processor,
    k=4,
    show=True,
)


# Run Experiment

In [0]:
from time import sleep
sleep(3600)

In [0]:
%cd /Workspace/Users/parkjea@pennmedicine.upenn.edu/TinyRecursiveModels

In [0]:
!pip install omegaconf

In [0]:
!python train_video.py --run_name clip_vid_trm_dh_notrunc_b8_c63 --L_cycles 6 --H_cycles 3 --halt_max_steps 8 --epochs 100 --hidden_size 256 --config_batch_size 8 --loss_option dense_head --lr_schedule --lr 1e-4

In [0]:
checkpoint_file="/Workspace/Users/parkjea@pennmedicine.upenn.edu/TinyRecursiveModels/checkpoints/clip_vid_trm_dh_notrunc_b8_c63/best.pth"

In [0]:
!python test_video.py --run_name test_clip_vid_trm_dh_notrunc_b8_c63 --model_pth $checkpoint_file --halt_max_steps 8 --hidden_size 256 --config_batch_size 8 --loss_option dense_head --lr_schedule  --feature_type clip

In [0]:
checkpoint_file="/Workspace/Users/parkjea@pennmedicine.upenn.edu/TinyRecursiveModels/checkpoints/i3d_vid_trm_dh_trunc_b8_c63_halt8_lr4/best.pth"

In [0]:
!echo $checkpoint_file

In [0]:
!python test_video_i3d.py --run_name test_i3d_vid_trm_dh_trunc_b8_c63_halt8_lr4 --model_pth $checkpoint_file --halt_max_steps 8 --hidden_size 256 --config_batch_size 8 --loss_option dense_head --lr_schedule  --feature_type i3d

In [0]:
!python train_video_i3d.py --run_name i3d_vid_trm_notrunc_b8_c42 --L_cycles 4 --H_cycles 2  --halt_max_steps 8 --epochs 100 --hidden_size 256 --config_batch_size 8 --loss_option dense_head --lr 1e-4 --lr_schedule